In [3]:
# Cell 1: Imports and env check
from dotenv import load_dotenv
import os

load_dotenv()
print("✅ Environment loaded")
print(f"API Key present: {'Yes' if os.getenv('GROQ_API_KEY') else 'No'}")

✅ Environment loaded
API Key present: Yes


In [ ]:
# Best models available on Groq free tier (as of 2025)
groq_models = {
    "best_overall":   "llama-3.3-70b-versatile",   # use this most often
    "fastest":        "llama-3.1-8b-instant",       # when speed matters
    "coding":         "qwen-2.5-coder-32b",         # for code generation
    "reasoning":      "deepseek-r1-distill-llama-70b", # for complex tasks
}

In [3]:
from langchain_groq import ChatGroq

llm_groq = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature=0.7
)

response = llm_groq.invoke("What is the capital of Gujarat?")
print(response)
print(response.content)
print(response.usage_metadata)
print(response.response_metadata)

content='The capital of Gujarat is Gandhinagar.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 42, 'total_tokens': 52, 'completion_time': 0.012403506, 'completion_tokens_details': None, 'prompt_time': 0.002363248, 'prompt_tokens_details': None, 'queue_time': 0.169822415, 'total_time': 0.014766754}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'finish_reason': 'stop', 'logprobs': None} id='run--019d3fe1-9644-7fb3-9162-c6f0b8a2f695-0' usage_metadata={'input_tokens': 42, 'output_tokens': 10, 'total_tokens': 52}
The capital of Gujarat is Gandhinagar.
{'input_tokens': 42, 'output_tokens': 10, 'total_tokens': 52}
{'token_usage': {'completion_tokens': 10, 'prompt_tokens': 42, 'total_tokens': 52, 'completion_time': 0.012403506, 'completion_tokens_details': None, 'prompt_time': 0.002363248, 'prompt_tokens_details': None, 'queue_time': 0.169822415, 'total_time': 0.014766754}, 'model_name': 'llama-3.3-70b-versa

In [3]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# Initialize the Groq model
llm = ChatGroq(model="llama-3.1-8b-instant")

# Create the agent
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='b65192af-b9fd-4aff-aae9-ada9f21a3bed'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hzc0jhvkn', 'function': {'arguments': '{"city":"sf"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 223, 'total_tokens': 237, 'completion_time': 0.033432663, 'completion_tokens_details': None, 'prompt_time': 0.141710843, 'prompt_tokens_details': None, 'queue_time': 0.047216058, 'total_time': 0.175143506}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d429f-1fbe-7583-bd2b-832b28980518-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': 'hzc0jhvkn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223,

In [16]:
# pip install -qU langchain langchain-groq
from langchain_groq import ChatGroq
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
import os

# Make sure your API key is set in your environment variables
# os.environ["GROQ_API_KEY"] = "your-groq-api-key"

# 1. We wrap your function in the @tool decorator
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

tools = [get_weather]

# 2. Initialize the Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# 3. Create a strict prompt template for the Agent
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "{input}"),
    # The agent_scratchpad is fundamentally required by Langchain AgentExecutors 
    # to store intermediate responses / tool thought processes:
    ("placeholder", "{agent_scratchpad}"),
])

# 4. Create the core agent that binds the LLM and the tools
agent = create_tool_calling_agent(
    llm=llm, 
    tools=tools, 
    prompt=prompt
)

# 5. Wrap it in an AgentExecutor which loops through the tools automatically
# Tip: set verbose=False if you don't want to see the terminal printout of its thoughts
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 6. Run the agent (AgentExecutor expects {"input": "..."} instead of a raw message list)
result = agent_executor.invoke({
    "input": "What is the weather in Patdi"
})

# Print just the final answer
print("\nFinal Output:", result["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Patdi'}`


It's always sunny in Patdi!
Invoking: `get_weather` with `{'city': 'Patdi'}`
responded: I'm not sure about the current weather in Patdi. Let me try to find out for you.



It's always sunny in Patdi!
Invoking: `get_weather` with `{'city': 'Patdi'}`
responded: I'm just kidding, I don't have have access to real-time information. Let me try that again.



It's always sunny in Patdi!I don't have any more information about the weather in Patdi.

> Finished chain.

Final Output: I don't have any more information about the weather in Patdi.
